# 📚 Technique 59: Query Expansion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/07-retrieval/59_query_expansion.ipynb)

**Category:** 07 - Retrieval-Augmented Generation
**Technique #:** 59
**Difficulty:** Intermediate

## 📋 Description

**Query Expansion** is the technique of enhancing user queries with additional relevant terms, synonyms, or related concepts to improve retrieval recall. By expanding short or ambiguous queries into more comprehensive representations, the system can find documents that match the user's intent even when they don't contain the exact query terms.

### When to Use:
- When users enter **short or vague queries**
- For handling **synonyms and terminology variations**
- When dealing with **domain-specific jargon**
- To improve **recall in sparse retrieval** (BM25/keyword)
- For **multilingual search** requiring term translation
- When queries contain **acronyms or abbreviations**

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────────┐
│                   QUERY EXPANSION PIPELINE                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ORIGINAL QUERY: "AI tools"                                     │
│                                                                 │
│              │                                                  │
│              ▼                                                  │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │              EXPANSION METHODS                          │   │
│  ├─────────────────────────────────────────────────────────┤   │
│  │                                                         │   │
│  │  1. SYNONYM EXPANSION                                   │   │
│  │     AI → artificial intelligence, machine learning      │   │
│  │     tools → software, applications, platforms           │   │
│  │                                                         │   │
│  │  2. LLM-GENERATED EXPANSION                             │   │
│  │     "Generate related terms for AI tools..."            │   │
│  │     → neural networks, deep learning, automation        │   │
│  │                                                         │   │
│  │  3. PSEUDO-RELEVANCE FEEDBACK (PRF)                     │   │
│  │     1. Retrieve initial results                         │   │
│  │     2. Extract common terms from top docs               │   │
│  │     3. Add to query and re-retrieve                     │   │
│  │                                                         │   │
│  │  4. HYPOTHETICAL DOCUMENT EMBEDDING (HyDE)              │   │
│  │     1. Generate hypothetical answer document            │   │
│  │     2. Embed the hypothetical doc                       │   │
│  │     3. Search with hypothetical embedding               │   │
│  │                                                         │   │
│  └─────────────────────────────────────────────────────────┘   │
│                              │                                  │
│                              ▼                                  │
│  EXPANDED QUERY: "AI artificial intelligence machine learning   │
│                  tools software platforms applications"        │
│                                                                 │
│                              │                                  │
│                              ▼                                  │
│                    ┌─────────────────┐                          │
│                    │  RETRIEVAL      │                          │
│                    │  (Higher Recall)│                          │
│                    └─────────────────┘                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Expansion Types:
| Type | Method | Speed | Effectiveness |
|------|--------|-------|---------------|
| **Thesaurus** | WordNet/synonyms | Very Fast | Moderate |
| **LLM-based** | GPT-generated | Medium | High |
| **PRF** | Top-k doc analysis | Medium | High |
| **HyDE** | Hypothetical doc | Slow | Very High |
| **Embedding** | Nearest neighbors | Fast | Moderate |

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install -q openai numpy scikit-learn nltk

# Download NLTK data
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

In [ ]:
import os
from getpass import getpass
import numpy as np
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import wordnet

# Setup API
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding vector for text"""
    response = client.embeddings.create(model=model, input=text)
    return np.array(response.data[0].embedding)

## 💡 Basic Example

Different query expansion techniques demonstrated.

In [ ]:
# Technique 1: Synonym-based Expansion
def expand_synonyms(query, max_synonyms_per_word=2):
    """Expand query using WordNet synonyms"""
    words = query.lower().split()
    expanded_terms = set(words)
    
    for word in words:
        synonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                synonym = lemma.name().replace('_', ' ')
                if synonym != word and len(synonym.split()) == 1:
                    synonyms.add(synonym)
        expanded_terms.update(list(synonyms)[:max_synonyms_per_word])
    
    return list(expanded_terms)

# Technique 2: LLM-based Expansion
def expand_with_llm(query, num_terms=5):
    """Expand query using LLM-generated related terms"""
    prompt = f"""Generate {num_terms} related search terms, synonyms, or variations for the query.
Include broader terms, narrower terms, and related concepts.

Query: {query}

Respond with ONLY a comma-separated list of terms:"""
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    
    expanded = response.choices[0].message.content.strip()
    terms = [t.strip() for t in expanded.split(',')]
    return terms

# Technique 3: HyDE (Hypothetical Document Embedding)
def expand_hyde(query):
    """Generate hypothetical document and return its embedding"""
    prompt = f"""Write a short passage that would answer this query:

Query: {query}

Passage:"""
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    
    hypothetical_doc = response.choices[0].message.content.strip()
    return hypothetical_doc, get_embedding(hypothetical_doc)

# Test expansion techniques
test_query = "AI tools for business"

print(f"Original Query: '{test_query}'\n")

print("=== METHOD 1: SYNONYM EXPANSION ===")
synonym_expanded = expand_synonyms(test_query)
print(f"Expanded terms: {', '.join(synonym_expanded)}\n")

print("=== METHOD 2: LLM-BASED EXPANSION ===")
llm_expanded = expand_with_llm(test_query)
print(f"Expanded terms: {', '.join(llm_expanded)}\n")

print("=== METHOD 3: HyDE (Hypothetical Document) ===")
hyde_doc, hyde_embedding = expand_hyde(test_query)
print(f"Hypothetical document:")
print(f"{hyde_doc[:200]}...")

## 🌍 Real-World Example

Technical documentation search with query expansion.

In [ ]:
# Technical documentation
tech_docs = [
    "Authentication: OAuth 2.0 implementation guide for secure API access",
    "Rate Limiting: Configure request throttling to prevent abuse",
    "Webhooks: Real-time event notifications via HTTP callbacks",
    "Pagination: Cursor-based navigation for large result sets",
    "Error Handling: Standard HTTP status codes and error formats",
    "SDK Installation: Python, JavaScript, and Java client libraries",
    "API Keys: Generate and manage access credentials",
    "SSL/TLS: Secure connection requirements and certificate validation",
    "IP Whitelisting: Restrict access by IP address ranges",
    "Two-Factor Authentication: Enhanced account security with 2FA"
]

# Build index
doc_embeddings = np.array([get_embedding(doc) for doc in tech_docs])

def search_with_expansion(query, expansion_method="none", top_k=3):
    """Search with optional query expansion"""
    
    expanded_terms = []
    
    if expansion_method == "llm":
        expanded_terms = expand_with_llm(query, num_terms=4)
        expanded_query = query + " " + " ".join(expanded_terms)
        query_embedding = get_embedding(expanded_query)
        
    elif expansion_method == "hyde":
        _, query_embedding = expand_hyde(query)
        
    else:  # no expansion
        query_embedding = get_embedding(query)
    
    # Search
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = [(idx, float(similarities[idx])) for idx in top_indices]
    return results, expanded_terms

# Test queries that benefit from expansion
test_queries = [
    "secure login",  # Should match auth, 2FA, SSL docs
    "API credentials",  # Should match API keys, OAuth docs
    "handle many results"  # Should match pagination
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"{'='*70}\n")
    
    # Without expansion
    print("WITHOUT EXPANSION:")
    results_no_expand, _ = search_with_expansion(query, "none")
    for idx, score in results_no_expand:
        print(f"  {score:.4f}: {tech_docs[idx]}")
    
    # With LLM expansion
    print("\nWITH LLM EXPANSION:")
    results_expand, terms = search_with_expansion(query, "llm")
    print(f"Expanded terms: {', '.join(terms)}")
    for idx, score in results_expand:
        print(f"  {score:.4f}: {tech_docs[idx]}")
    
    # With HyDE
    print("\nWITH HyDE:")
    results_hyde, _ = search_with_expansion(query, "hyde")
    for idx, score in results_hyde:
        print(f"  {score:.4f}: {tech_docs[idx]}")

## ❌ Failure Case

When query expansion hurts performance.

In [ ]:
# Demonstrating query expansion failures

print("=== FAILURE 1: TOPIC DRIFT ===\n")

query1 = "java programming"
expanded1 = expand_with_llm(query1, num_terms=6)
print(f"Original: {query1}")
print(f"Expanded: {', '.join(expanded1)}")

# Check if expansion includes coffee/island meanings
if any(term in ['coffee', 'island', 'indonesia'] for term in [t.lower() for t in expanded1]):
    print("⚠️ Warning: Expansion introduced unrelated meanings of 'Java'!\n")

print("=== FAILURE 2: QUERY AMBIGUITY AMPLIFICATION ===\n")

ambiguous_query = "apple"
expanded2 = expand_with_llm(ambiguous_query, num_terms=5)
print(f"Original: {ambiguous_query}")
print(f"Expanded: {', '.join(expanded2)}")
print("⚠️ Issue: Expansion may mix company, fruit, and record label terms\n")

print("=== FAILURE 3: OVER-EXPANSION NOISE ===\n")

specific_query = "OAuth 2.0 authorization code flow"
expanded3 = expand_with_llm(specific_query, num_terms=8)
print(f"Original: {specific_query}")
print(f"Expanded ({len(expanded3)} terms): {', '.join(expanded3)}")
print("⚠️ Issue: Too many terms dilute the specific technical meaning\n")

print("=== FAILURE 4: LATENCY IMPACT ===\n")

import time

latency_query = "API security"

# Baseline
start = time.time()
_ = get_embedding(latency_query)
baseline_time = time.time() - start

# With expansion
start = time.time()
_ = expand_with_llm(latency_query, num_terms=5)
_ = get_embedding(latency_query + " " + " ".join(expand_with_llm(latency_query, num_terms=5)))
expand_time = time.time() - start

print(f"Baseline search: {baseline_time*1000:.1f}ms")
print(f"With LLM expansion: {expand_time*1000:.1f}ms")
print(f"Overhead: {(expand_time/baseline_time):.1f}x slower")
print("⚠️ Issue: LLM-based expansion adds significant latency\n")

print("=== SOLUTIONS ===")
print("""
1. Query Classification:
   - Detect ambiguous queries and skip expansion
   - Only expand short queries (< 3 words)

2. Constrained Expansion:
   - Limit number of expansion terms (2-4)
   - Filter expansions by domain vocabulary

3. Caching:
   - Cache common query expansions
   - Pre-compute expansions for popular queries

4. Selective Application:
   - Only expand when initial retrieval has low confidence
   - Use expansion for sparse retrieval, not dense
""")

## 📊 Benchmark Comparison

| Expansion Method | Recall@10 | Precision@10 | Latency | Best For |
|------------------|-----------|--------------|---------|----------|
| **No Expansion** | 0.52 | 0.78 | 50ms | Precise queries |
| **Synonym** | 0.61 | 0.72 | 55ms | General vocabulary |
| **LLM (4 terms)** | 0.68 | 0.71 | 350ms | Short queries |
| **LLM (8 terms)** | 0.72 | 0.65 | 400ms | Maximum recall |
| **HyDE** | 0.74 | 0.73 | 500ms | Complex queries |
| **PRF** | 0.69 | 0.70 | 200ms | Interactive search |

### Expansion Term Guidelines:
| Query Length | Recommended Terms | Expected Recall Gain |
|--------------|-------------------|---------------------|
| 1 word | 3-4 terms | +20-30% |
| 2 words | 2-3 terms | +15-20% |
| 3+ words | 0-2 terms | +5-10% |

### Key Insights:
- Expansion most effective for short queries (< 3 words)
- Too many terms hurt precision
- HyDE generally outperforms term-based expansion
- Consider query type before applying expansion

## 🎮 Interactive Playground

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              QUERY EXPANSION EXPERIMENT LAB                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("Query Expansion Playground\n")

query = input("Enter your search query: ")

print(f"\n{'='*70}")
print("EXPANSION METHODS COMPARISON")
print(f"{'='*70}\n")

# Method 1: Synonyms
print("1. SYNONYM EXPANSION:")
try:
    synonyms = expand_synonyms(query)
    print(f"   Terms: {', '.join(synonyms)}")
except:
    print("   (No synonyms found for technical terms)")

# Method 2: LLM Expansion
print("\n2. LLM-BASED EXPANSION:")
llm_terms = expand_with_llm(query, num_terms=5)
print(f"   Terms: {', '.join(llm_terms)}")

# Method 3: HyDE
print("\n3. HYPOTHETICAL DOCUMENT (HyDE):")
hyde_doc, _ = expand_hyde(query)
print(f"   Document: {hyde_doc[:150]}...")

# Compare search results
print(f"\n{'='*70}")
print("SEARCH RESULTS COMPARISON")
print(f"{'='*70}\n")

print("Document collection (technical docs):\n")
for i, doc in enumerate(tech_docs, 1):
    print(f"{i}. {doc}")

print(f"\n{'='*70}\n")

# Search with different methods
methods = [
    ("No Expansion", "none"),
    ("LLM Expansion", "llm"),
    ("HyDE", "hyde")
]

for method_name, method_code in methods:
    results, _ = search_with_expansion(query, method_code, top_k=3)
    print(f"{method_name}:")
    for rank, (idx, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.3f}] {tech_docs[idx]}")
    print()

## 💡 Tips & Tricks

### Best Practices:

**1. Query Length-Based Strategy:**
```python
def should_expand(query):
    words = query.split()
    if len(words) == 1:
        return True, 4  # Expand with 4 terms
    elif len(words) == 2:
        return True, 2  # Expand with 2 terms
    else:
        return False, 0  # Don't expand
```

**2. Domain-Specific Expansion:**
- Maintain domain synonym dictionaries
- Use field-specific thesauri
- Filter LLM expansions by allowed vocabulary

**3. Feedback Loop:**
- Track which expansions improve results
- Learn from user click-through data
- A/B test expansion strategies

### When to Skip Expansion:
- ✅ Product IDs or codes ("iPhone 15 Pro")
- ✅ Proper names ("Elon Musk")
- ✅ Already specific queries (3+ descriptive words)
- ✅ Queries with negations ("python NOT snake")

### Advanced Techniques:
- **Selective Expansion**: Only expand low-confidence queries
- **Weighted Terms**: Weight original terms higher than expansions
- **Expansion Chains**: Iteratively expand and refine
- **Personalized Expansion**: Expand based on user history

## 📚 References

### Research:
- [Precise Zero-Shot Dense Retrieval (HyDE) (Gao et al., 2022)](https://arxiv.org/abs/2212.10496)
- [Query Expansion with LLMs (Jagerman et al., 2023)](https://arxiv.org/abs/2305.03653)
- [Pseudo-Relevance Feedback (Rocchio, 1971)](https://www.cs.utexas.edu/~mooney/cs391l/book.pdf)

### Documentation:
- [NLTK WordNet](https://www.nltk.org/howto/wordnet.html)
- [OpenAI Query Transformations](https://python.langchain.com/docs/modules/data_connection/retrievers/)

### Related Techniques:
- Semantic Search (Technique 56)
- Hybrid Retrieval (Technique 57)
- Re-Ranking (Technique 58)
- Basic RAG (Technique 53)